In [6]:
# 1. Importar bibliotecas
import os
import pandas as pd
import sqlite3

# 2. Garantir que a pasta data existe
os.makedirs("../data", exist_ok=True)

# 3. Criar (ou abrir) o banco de dados
db_path = "../data/nubank.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
print(f"Banco criado em: {db_path}")

Banco criado em: ../data/nubank.db


## Carregamento das planilhas de 2021 – Aba CC1

In [7]:
import os, pandas as pd, sqlite3

conn = sqlite3.connect("../data/nubank.db")
raw_root = "../data/raw/2021"

map_periodo_2021 = {
    "anexo-pilar3-circular-3930-q2-2021.xlsx": "2021-06-30",
    "anexo-pilar3-circular-3930-q4-2021.xlsx": "2021-12-31"
}

for file in os.listdir(raw_root):
    if file.endswith(".xlsx") and not file.startswith("~$"):
        prefix = file.replace(".xlsx", "")
        xls = pd.ExcelFile(os.path.join(raw_root, file))
        if "CC1" in xls.sheet_names:
            df = pd.read_excel(xls, sheet_name="CC1", skiprows=3)
            df = df.iloc[:, :3]
            df.columns = ["codigo", "descricao", "valor"]
            # Usar a data mapeada
            df["periodo"] = map_periodo_2021.get(file, None)
            table_name = f"{prefix}_CC1".replace(" ", "_")
            df.to_sql(table_name, conn, if_exists="replace", index=False)
            print(f"Tabela {table_name} criada com {len(df)} registros.")

conn.close()

Tabela anexo-pilar3-circular-3930-q2-2021_CC1 criada com 88 registros.
Tabela anexo-pilar3-circular-3930-q4-2021_CC1 criada com 88 registros.


## Carregamento da planilha de 2022 (T1) – Aba KM1 (estrutura diferente)

In [8]:
import pandas as pd, sqlite3

path = "../data/raw/2022/anexo-pilar3-circular-3930-q1-2022.xlsx"

df_raw = pd.read_excel(path, sheet_name="KM1", header=None)

periodos = list(df_raw.iloc[2, 2:].dropna().astype(str))
df_data = df_raw.iloc[4:, 0:2+len(periodos)].copy()
df_data.columns = ["codigo", "descricao"] + periodos

df_long = pd.melt(df_data,
                  id_vars=["codigo","descricao"],
                  value_vars=periodos,
                  var_name="periodo",
                  value_name="valor")

df_long = df_long.dropna(subset=["descricao","valor"])
df_long["valor"] = pd.to_numeric(df_long["valor"], errors="coerce")

# Forçar período único para Q1 2022
df_long["periodo"] = "2022-03-31"

conn = sqlite3.connect("../data/nubank.db")
df_long.to_sql("anexo_pilar3_q1_2022_KM1", conn, if_exists="replace", index=False)
print(f"Tabela anexo_pilar3_q1_2022_KM1 criada com {len(df_long)} registros.")
conn.close()

Tabela anexo_pilar3_q1_2022_KM1 criada com 70 registros.


## Carregamento das planilhas de 2022 (T2) a 2025 – Aba KM1

In [9]:
import os, pandas as pd, sqlite3

def parse_excel_matrix(path, sheet):
    df_raw = pd.read_excel(path, sheet_name=sheet, header=None)
    letters = df_raw.iloc[5, 4:].tolist()
    dates   = df_raw.iloc[6, 4:].tolist()
    periodos = [f"{l}_{d}" for l, d in zip(letters, dates)]
    df_data = df_raw.iloc[8:, [2,3] + list(range(4, 4+len(periodos)))]
    df_data.columns = ["codigo", "descricao"] + periodos
    df_long = df_data.melt(id_vars=["codigo","descricao"], var_name="periodo", value_name="valor")
    df_long = df_long.dropna(subset=["descricao","valor"])
    df_long["periodo"] = df_long["periodo"].str.extract(r'(\d{4}-\d{2}-\d{2})')
    df_long["valor"] = pd.to_numeric(df_long["valor"], errors="coerce")
    return df_long

conn = sqlite3.connect("../data/nubank.db")

for root, dirs, files in os.walk("../data/raw"):
    for file in files:
        if file.endswith(".xlsx") and not file.startswith("~$"):
            if "2022" in root and "q1" in file.lower():
                continue  # pula o 1º trimestre de 2022
            if "2021" in root:
                continue  # pula 2021 (já tratado)
            path = os.path.join(root, file)
            df_long = parse_excel_matrix(path, "KM1")
            table_name = f"{file.replace('.xlsx','')}_KM1".replace(" ", "_")
            df_long.to_sql(table_name, conn, if_exists="replace", index=False)
            print(f"Tabela {table_name} criada com {len(df_long)} registros.")

conn.close()

Tabela anexo-pilar3-circular-3930-q2-2022_KM1 criada com 70 registros.
Tabela anexo-pilar3-circular-3930-q3-2022_KM1 criada com 60 registros.
Tabela anexo-pilar3-circular-3930-q4-2022_KM1 criada com 70 registros.
Tabela anexo-pilar3-circular-3930-q1-2023_KM1 criada com 50 registros.
Tabela anexo-pilar3-circular-3930-q2-2023_KM1 criada com 70 registros.
Tabela anexo-pilar3-circular-3930-q3-2023_KM1 criada com 12 registros.
Tabela anexo-pilar3-circular-3930-q4-2023_KM1 criada com 28 registros.
Tabela anexo-pilar3-circular-3930-q1-2024_KM1 criada com 42 registros.
Tabela anexo-pilar3-circular-3930-q2-2024_KM1 criada com 56 registros.
Tabela anexo-pilar3-circular-3930-q3-2024_KM1 criada com 70 registros.
Tabela anexo-pilar3-circular-3930-q4-2024_KM1 criada com 60 registros.
Tabela anexo-pilar3-circular-3930-q1-2025__1__KM1 criada com 60 registros.
Tabela anexo-pilar3-circular-3930-q2-2025__1__KM1 criada com 115 registros.
Tabela anexo-pilar3-circular-3930-q3-2025__1__KM1 criada com 115 reg

## Consolidação das 18 tabelas

In [10]:
import sqlite3
import pandas as pd

# Conectar ao banco
conn = sqlite3.connect("../data/nubank.db")

# Listar todas as tabelas criadas
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tabelas = [t[0] for t in cursor.fetchall()]

# Filtrar apenas as tabelas relevantes (CC1 de 2021, KM1 de 2022–2025)
tabelas_relevantes = [t for t in tabelas if ("CC1" in t or "KM1" in t)]

print("Tabelas relevantes:", tabelas_relevantes)

# Consolidar todas em um único DataFrame
dfs = []
for t in tabelas_relevantes:
    df = pd.read_sql_query(f"SELECT codigo, descricao, periodo, valor FROM '{t}'", conn)
    df["origem_tabela"] = t  # opcional: guardar de qual tabela veio
    dfs.append(df)

df_hist = pd.concat(dfs, ignore_index=True)

# Salvar tabela consolidada
df_hist.to_sql("historico_capital", conn, if_exists="replace", index=False)
print(f"Tabela 'historico_capital' criada com {len(df_hist)} registros.")

conn.close()


Tabelas relevantes: ['anexo-pilar3-circular-3930-q2-2021_CC1', 'anexo-pilar3-circular-3930-q4-2021_CC1', 'anexo_pilar3_q1_2022_KM1', 'anexo-pilar3-circular-3930-q2-2022_KM1', 'anexo-pilar3-circular-3930-q3-2022_KM1', 'anexo-pilar3-circular-3930-q4-2022_KM1', 'anexo-pilar3-circular-3930-q1-2023_KM1', 'anexo-pilar3-circular-3930-q2-2023_KM1', 'anexo-pilar3-circular-3930-q3-2023_KM1', 'anexo-pilar3-circular-3930-q4-2023_KM1', 'anexo-pilar3-circular-3930-q1-2024_KM1', 'anexo-pilar3-circular-3930-q2-2024_KM1', 'anexo-pilar3-circular-3930-q3-2024_KM1', 'anexo-pilar3-circular-3930-q4-2024_KM1', 'anexo-pilar3-circular-3930-q1-2025__1__KM1', 'anexo-pilar3-circular-3930-q2-2025__1__KM1', 'anexo-pilar3-circular-3930-q3-2025__1__KM1', 'anexo-pilar3-circular-3930-q4-2025_KM1']
Tabela 'historico_capital' criada com 1239 registros.
